# Evaluation run: the whole prompt set in one go

Runs every prompt of `eval/prompts.csv` for one `PART`, and keeps going when a prompt fails
(Gemini 503/429, CUDA out of memory, ...): it waits, tries that prompt once more, then records `ERROR` and moves on.
`summary_<PART>.csv` is rewritten after every prompt, so nothing finished is lost.

**How to run (best: in the background)**
1. Same one-time setup as `egig_kaggle.ipynb`: GPU T4 x2, Internet On, Secrets `GEMINI_API_KEY`, `WIKI_CONTACT`, `GITHUB_TOKEN`
   (each person uses **their own** Gemini key, so the free quota is not shared).
2. Edit the settings cell below: `PART` = `"A"`, `"B"` or `"ALL"` (one person runs everything);
   everybody uses the **same** `GIT_REF`. The prompts come from `eval/prompts.csv` in the repo: nothing to upload.
3. **Save Version -> Save & Run All (Commit)**. It runs without the browser open (up to 12 h).
4. When it is done: open the version -> **Output** -> download `eval_<split>_<part>.zip`.
   Send it to whoever merges the results (Google Drive / LINE). Do not push images to git.

**Testing only the image generator** (`MAX_ATTEMPTS = 1`): every prompt is drawn once, with no repair loop,
so the verifier cannot change any picture. Output goes to `eval_<split>_gen1` (kept apart from full runs).
Label those images on the laptop with `scripts/make_review.py` and compare with the first attempts of the old run;
re-check them there with `scripts/reverify.py` if you want the new verifier's verdicts.

Interactive instead? Run all cells top to bottom. Running the batch cell again skips prompts that are already PASS/FAIL
and redoes `ERROR` / `NOT_VERIFIED` ones.

In [ ]:
# Settings: the only cell you need to edit
PART = "A"            # "A" (8 prompts), "B" (7 prompts), or "ALL" (all 15 of the split in one run)
SPLIT = "dev"         # "dev" while fixing; "holdout" only once at the very end
ONLY = []             # e.g. ["d03_shiba_akita"] to (re)run just these ids; [] = all of this part
GIT_REF = "main"      # the code version: everybody must use the SAME one (a branch, tag or full commit hash)
MAX_ATTEMPTS = 3      # 3 = full pipeline with the repair loop; 1 = test only the image generator (no loop, the
                      #     verifier cannot change the picture; output in eval_<split>_gen1)


In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# 1) Get the code into /kaggle/working/egig_project
import glob, os, shutil, subprocess

REPO_URL = "https://github.com/itzPEKOS/entity-grounded-image-generation.git"  # empty = use the uploaded dataset
WORK = "/kaggle/working/egig_project"

if REPO_URL:
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    try:  # private repo: add a GitHub token as the Kaggle secret GITHUB_TOKEN
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN").strip()
    except Exception:
        token = ""  # public repo: no token needed
    if token:  # token goes in an HTTP header via env vars: never in the URL, .git/config, or error output
        import base64
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {basic}")
    if not os.path.exists(os.path.join(WORK, ".git")):
        proc = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], env=env, capture_output=True, text=True)
    else:
        proc = subprocess.CompletedProcess([], 0, "", "")
    if proc.returncode == 0:  # then move to exactly GIT_REF
        proc = subprocess.run(["git", "-C", WORK, "fetch", "--depth", "1", "origin", GIT_REF],
                              env=env, capture_output=True, text=True)
    if proc.returncode == 0:
        proc = subprocess.run(["git", "-C", WORK, "checkout", "-q", "--detach", "FETCH_HEAD"],
                              capture_output=True, text=True)
    if proc.returncode != 0:
        hint = ("GITHUB_TOKEN was found but GitHub refused it: check it can read this repo and has not expired."
                if token else
                "Repo is private (or the URL is wrong). Make it public, or add a GitHub token as the Kaggle "
                "secret GITHUB_TOKEN, attach it to this notebook, and run this cell again.")
        raise RuntimeError(proc.stderr.strip() + "\n\n-> " + hint)
    print(subprocess.run(["git", "-C", WORK, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
else:
    hits = glob.glob("/kaggle/input/**/egig/__init__.py", recursive=True)
    assert hits, "No code found: upload dist/egig_code.zip as a Dataset and Add Input it to this notebook."
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), WORK, dirs_exist_ok=True)
os.chdir(WORK)
print("code in", WORK, "->", sorted(os.listdir(WORK)))
COMMIT = subprocess.run(["git", "-C", WORK, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code version:", COMMIT or "(uploaded dataset)")


In [ ]:
# 2) Packages. torch+CUDA is preinstalled; upgrade diffusers AND transformers together so their
#    huggingface_hub requirements stay compatible (tested with diffusers 0.43 + transformers 5.24)
!pip install -q -U "diffusers>=0.32" "transformers>=4.45" "accelerate>=0.30" "google-genai>=1.0" python-dotenv
import torch, transformers, diffusers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| diffusers", diffusers.__version__,
      "| GPUs:", torch.cuda.device_count())

In [ ]:
# 3) Secrets -> environment variables
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ["GEMINI_API_KEY", "WIKI_CONTACT"]:
    try:
        os.environ[name] = secrets.get_secret(name)
        print(name, "loaded")
    except Exception:
        print(name, "MISSING: add it under Add-ons > Secrets and attach it to this notebook")
os.environ["OUTPUT_DIR"] = "/kaggle/working/outputs"

In [ ]:
# Build the agents once; models stay loaded for every prompt
import sys
sys.path.insert(0, WORK)
from egig.config import Settings
from egig.orchestrator import build_orchestrator
from egig.evaluation import load_prompts, run_batch

prompts = load_prompts(os.path.join(WORK, "eval", "prompts.csv"), split=SPLIT, part=None if PART == "ALL" else PART,
                       only=ONLY or None)
assert prompts, f"no prompts for split={SPLIT!r} part={PART!r} only={ONLY}"
for p in prompts:
    print(f"{p.id:<24} {p.prompt}")
settings = Settings.from_env(max_attempts=MAX_ATTEMPTS)
print("max_attempts =", settings.max_attempts)
orch = build_orchestrator(settings)

In [ ]:
# Run the whole part. Takes roughly 4-7 minutes per prompt (more with 3+ animals).
SUBDIR = f"eval_{SPLIT}" + ("_gen1" if MAX_ATTEMPTS == 1 else "")
OUT = os.path.join(os.environ["OUTPUT_DIR"], SUBDIR)
ZIP = f"/kaggle/working/{SUBDIR}_{PART}"
try:
    rows = run_batch(orch, prompts, subdir=SUBDIR, summary_name=f"summary_{PART}.csv", commit=COMMIT,
                     retry_wait_s=120, outage_wait_s=600,
                     rerun_not_verified=MAX_ATTEMPTS > 1)  # generator-only: Gemini down is not a reason to redraw
finally:  # zip even if the cell is stopped half way: every finished prompt is kept
    if os.path.isdir(OUT):
        print("zip:", shutil.make_archive(ZIP, "zip", OUT))

In [ ]:
# Summary of this part
import pandas as pd
pd.read_csv(os.path.join(OUT, f"summary_{PART}.csv"))[["id", "status", "final_attempt", "attempts", "critical",
                                                      "major_rate", "gemini_calls", "seconds", "error"]]